# Acuifero + Vigia ? Live Hardware Replay Demo

This public-Kaggle-ready notebook is a **hardware replay demo**, not a live Raspberry Pi stream.

First-screen honesty checklist:

- This is a hardware replay demo.
- It is not a live Raspberry Pi stream.
- `RUN_GEMMA = False` by default.
- The notebook does not execute LiteRT-LM or Gemma by default.
- Raspberry Pi 5 + LiteRT-LM proof is documented separately in the repository evidence, benchmark card, and demo video artifacts.

It lets judges run the Acuifero + Vigia alert orchestration without Raspberry Pi, Docker, backend, frontend, internet downloads, or GPU access. The notebook uses small captured/synthetic-safe fixtures and a golden output to demonstrate the **same data contract**, **same prompt contract**, and **same alert orchestration** expected from the deployed system.

Default mode is `RUN_GEMMA = False`. In that mode this notebook does **not** run LiteRT-LM or live Gemma inference. The real Raspberry Pi 5 8GB + LiteRT-LM hardware proof is documented separately in the repository evidence, benchmark card, and demo video artifacts.

The SINAGIR/CAP-shaped output shown here is a preview only. Nothing is submitted to SINAGIR production endpoints.

## 1. Fixture Discovery

Path priority:

1. `/kaggle/input/acuifero-vigia-fixtures/`
2. `./fixtures/kaggle_live_demo/`
3. `../fixtures/kaggle_live_demo/`

The notebook intentionally does not download files from the internet.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
import traceback
from pathlib import Path
from pprint import pprint

try:
    from IPython.display import HTML, Image, Markdown, display
except Exception:
    class HTML:
        def __init__(self, data): self.data = data
        def __repr__(self): return str(self.data)
    class Markdown:
        def __init__(self, data): self.data = data
        def __repr__(self): return str(self.data)
    class Image:
        def __init__(self, filename=None, width=None):
            self.filename = filename
            self.width = width
        def __repr__(self): return f"Image(filename={self.filename!r}, width={self.width!r})"
    def display(obj):
        print(obj)

RUN_GEMMA = False

CANDIDATE_FIXTURE_ROOTS = [
    Path("/kaggle/input/acuifero-vigia-fixtures"),
    Path("./fixtures/kaggle_live_demo"),
    Path("../fixtures/kaggle_live_demo"),
]

REQUIRED_RELATIVE_FILES = [
    "manifest.json",
    "sensor_readings.csv",
    "vigia_reports.jsonl",
    "frames/00_nominal.jpg",
    "frames/01_vigilance.jpg",
    "frames/02_urgent.jpg",
    "frames/03_critical.jpg",
    "transcripts/report_01.txt",
    "expected_outputs/alert_trace.json",
]


def find_fixture_root() -> Path:
    for root in CANDIDATE_FIXTURE_ROOTS:
        if root.exists() and all((root / rel).exists() for rel in REQUIRED_RELATIVE_FILES):
            return root
    checked = "\n".join(f"- {root.resolve()}" for root in CANDIDATE_FIXTURE_ROOTS)
    required = "\n".join(f"- {rel}" for rel in REQUIRED_RELATIVE_FILES)
    raise FileNotFoundError(
        "Acuifero + Vigia live-demo fixtures were not found.\n\n"
        f"Checked roots:\n{checked}\n\nRequired files at the fixture root:\n{required}"
    )

FIXTURE_ROOT = find_fixture_root()
print(f"Fixture root: {FIXTURE_ROOT.resolve()}")
print(f"RUN_GEMMA={RUN_GEMMA} (default: golden fixture replay, no live model inference)")


## 2. Fixture Validation

This cell checks that every declared fixture exists, the CSV has the expected columns, JSONL parses line by line, and the golden alert has the required decision fields.

In [ ]:
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_number}: {exc}") from exc
    return rows


def load_csv(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

manifest = load_json(FIXTURE_ROOT / "manifest.json")
manifest_files = manifest.get("files", {})
manifest_paths: list[str] = []
for value in manifest_files.values():
    if isinstance(value, str):
        manifest_paths.append(value)
    elif isinstance(value, list):
        manifest_paths.extend(item for item in value if isinstance(item, str))

missing_manifest_paths = [rel for rel in manifest_paths if not (FIXTURE_ROOT / rel).exists()]
if missing_manifest_paths:
    raise FileNotFoundError(f"Manifest references missing paths: {missing_manifest_paths}")

sensor_rows = load_csv(FIXTURE_ROOT / "sensor_readings.csv")
reports = load_jsonl(FIXTURE_ROOT / "vigia_reports.jsonl")
transcript_text = (FIXTURE_ROOT / "transcripts/report_01.txt").read_text(encoding="utf-8").strip()
alert_trace = load_json(FIXTURE_ROOT / "expected_outputs/alert_trace.json")

required_columns = set(manifest.get("required_columns", []))
actual_columns = set(sensor_rows[0].keys()) if sensor_rows else set()
missing_columns = sorted(required_columns - actual_columns)
if missing_columns:
    raise ValueError(f"sensor_readings.csv missing columns: {missing_columns}")

final_alert = alert_trace.get("final_alert", {})
if not final_alert.get("level"):
    raise ValueError("alert_trace.json final_alert.level is required")
if not isinstance(final_alert.get("evidence"), list) or not final_alert["evidence"]:
    raise ValueError("alert_trace.json final_alert.evidence must be a non-empty list")

print("Fixture validation passed")
print(f"sensor_rows={len(sensor_rows)} reports={len(reports)} frames={len(manifest_files.get('frames', []))}")
print(f"golden_output_mode={alert_trace.get('runtime', {}).get('mode')}")
print(f"notebook_executes_model={alert_trace.get('runtime', {}).get('notebook_executes_model')}")


## 3. Display Inputs

The judge can inspect the replay inputs directly: sensor readings, Vigia reports, curated frames, and transcript text.

In [ ]:
def show_table(rows: list[dict], columns: list[str] | None = None, limit: int = 10) -> None:
    rows = rows[:limit]
    if columns is None and rows:
        columns = list(rows[0].keys())
    columns = columns or []
    header = "".join(f"<th>{col}</th>" for col in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{row.get(col, '')}</td>" for col in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Sensor readings")
show_table(sensor_rows)

print("Vigia reports")
show_table(reports, columns=["report_id", "timestamp", "reporter_role", "offline_created", "transcript"])

print("Transcript")
display(Markdown(f"> {transcript_text}"))

for rel in manifest_files.get("frames", []):
    display(Markdown(f"**{rel}**"))
    display(Image(filename=str(FIXTURE_ROOT / rel), width=420))


## 4. Deterministic Replay

This replay derives water states from fixture readings. It is deterministic and does not require model inference.

In [ ]:
def classify_node_state(row: dict) -> str:
    water = float(row["water_level_cm"])
    reference = float(row["reference_line_cm"])
    urgent = float(row["urgent_line_cm"])
    critical = float(row["critical_line_cm"])
    if water >= critical:
        return "critical"
    if water >= urgent:
        return "urgent"
    if water >= reference:
        return "vigilance"
    return "nominal"


def evidence_from_row(row: dict) -> dict:
    state = classify_node_state(row)
    return {
        "timestamp": row["timestamp"],
        "source": row["source"],
        "frame": row["frame_path"],
        "water_level_cm": row["water_level_cm"],
        "reference_line_cm": row["reference_line_cm"],
        "urgent_line_cm": row["urgent_line_cm"],
        "critical_line_cm": row["critical_line_cm"],
        "derived_state": state,
    }

node_evidence = [evidence_from_row(row) for row in sensor_rows]
show_table(node_evidence)

highest_state = node_evidence[-1]["derived_state"]
print(f"Latest derived node state: {highest_state}")


## 5. Gemma Replay

Default mode loads the golden fixture output. It does **not** claim live LiteRT-LM or live Gemma inference.

If `RUN_GEMMA=True`, the notebook makes a conservative local-only attempt through the repo's LiteRT adapter when the repo, Python package, LiteRT runtime, and model file are already present. It does not download anything. If anything is missing or fails, the notebook prints the failure and continues with the golden output.

In [ ]:
SYSTEM_PROMPT = """You are Acuifero + Vigia, an auditable flood-risk alert orchestrator.
Return strict JSON for final_alert and sinagir_ready_export.
Do not claim production SINAGIR submission.
"""

USER_PROMPT = json.dumps(
    {
        "mode": "hardware_replay",
        "sensor_readings": sensor_rows,
        "vigia_reports": reports,
        "transcript": transcript_text,
        "contract": "same data contract, same prompt contract, same alert orchestration; cached/golden inference by default",
    },
    ensure_ascii=False,
    indent=2,
)


def try_local_litert_runtime(system_prompt: str, user_prompt: str) -> dict:
    repo_backend_src = Path.cwd() / "backend" / "src"
    if repo_backend_src.exists():
        sys.path.insert(0, str(repo_backend_src))
    try:
        from acuifero_vigia.adapters.litert_node import LiteRTNodeRuntime
    except Exception as exc:
        raise RuntimeError(f"Repo LiteRT adapter is not importable in this notebook environment: {exc}") from exc

    runtime = LiteRTNodeRuntime()
    health = runtime.health()
    if not health.reachable:
        raise RuntimeError(f"Local LiteRT runtime is not reachable: {health.detail}")

    payload = runtime.generate_json(system_prompt, user_prompt, max_tokens=768)
    if not isinstance(payload, dict):
        raise RuntimeError("Local LiteRT runtime returned no parseable JSON")
    payload.setdefault("runtime", {})
    payload["runtime"].update(
        {
            "mode": "live_local_litert_inference",
            "notebook_executes_model": True,
            "model_family": "Gemma 4",
            "prompt_version": "kaggle-live-demo-v1",
            "disclaimer": "Live local inference ran only because a local model/runtime was already present; no internet download was performed.",
        }
    )
    return payload

selected_alert_trace = alert_trace
live_inference_error = None

if RUN_GEMMA:
    try:
        selected_alert_trace = try_local_litert_runtime(SYSTEM_PROMPT, USER_PROMPT)
        print("MODE: live_local_litert_inference")
    except Exception as exc:
        live_inference_error = f"{type(exc).__name__}: {exc}"
        print("MODE: golden_fixture_output")
        print("Live local inference was requested but unavailable; continuing with golden fixture output.")
        print(live_inference_error)
else:
    print("MODE: golden_fixture_output")
    print("RUN_GEMMA=False, so this notebook is not running LiteRT-LM or live Gemma inference.")

runtime = selected_alert_trace.get("runtime", {})
pprint(runtime)


## 6. Fused Alert Output

This section displays the final alert object, evidence, recommended action, and SINAGIR/CAP-shaped preview with the required disclaimer.

In [ ]:
final_alert = selected_alert_trace["final_alert"]
sinagir_preview = selected_alert_trace["sinagir_ready_export"]

level = str(final_alert.get("level", "unknown")).upper()
score = final_alert.get("score")
reasoning_summary = final_alert.get("reasoning_summary", "")
recommended_action = final_alert.get("recommended_action", "")

badge_color = {
    "NOMINAL": "#3f8f5f",
    "VIGILANCE": "#c29b2e",
    "URGENT": "#c56b2f",
    "CRITICAL": "#b4232d",
}.get(level, "#475569")

display(HTML(f"""
<div style="border:1px solid #d1d5db;border-radius:8px;padding:16px;max-width:860px">
  <div style="font-size:14px;color:#475569">Hardware replay result</div>
  <div style="font-size:28px;font-weight:700;color:{badge_color}">Final alert: {level}</div>
  <div style="font-size:18px;margin:4px 0 12px 0">Score: {score}</div>
  <p><strong>Reasoning summary:</strong> {reasoning_summary}</p>
  <p><strong>Recommended action:</strong> {recommended_action}</p>
</div>
"""))

print("Evidence")
show_table(final_alert.get("evidence", []), columns=["source", "state", "detail"])

print("SINAGIR/CAP-shaped export preview")
print(sinagir_preview.get("disclaimer"))
pprint(sinagir_preview)


## 7. Save Output

On Kaggle, the output is written to `/kaggle/working/acuifero_vigia_alert_trace.json`. Locally, the notebook writes to the current working directory if `/kaggle/working` does not exist.

In [ ]:
output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
output_path = output_dir / "acuifero_vigia_alert_trace.json"
with output_path.open("w", encoding="utf-8") as handle:
    json.dump(selected_alert_trace, handle, indent=2, ensure_ascii=False)

print(f"Saved alert trace to: {output_path}")
print("Kaggle expected path: /kaggle/working/acuifero_vigia_alert_trace.json")


## Acceptance Notes

- This notebook succeeds without Raspberry Pi, backend, frontend, Docker, internet downloads, or GPU.
- Default mode uses `golden_fixture_output`, not live Gemma inference.
- The demonstrated scope is hardware replay: same data contract, same prompt contract, same alert orchestration.
- Real Raspberry Pi 5 + LiteRT-LM evidence is documented separately in the repository.
- SINAGIR/CAP output is a preview only and is not submitted to production endpoints.